# Lecture 19: Word Embeddings --- Meaning as Vectors

Last time, we built Markov text generators that capture local word patterns. But they don't understand *meaning*. Today, we explore **word embeddings**: representing words as vectors where similar meanings are close together.

## Setup

We'll use a pre-trained **GloVe** model trained on Wikipedia and Gigaword (news). Each word is represented as a 100-dimensional vector.

**Note:** The model file is ~130 MB and downloads on first use. All words are lowercase.

In [ ]:
import sys, subprocess
try:
    import gensim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gensim'])

import numpy as np

# Load the pre-trained GloVe model
import gensim.downloader as api
print("Loading GloVe model (downloads ~130 MB on first run)...")
model = api.load('glove-wiki-gigaword-100')
print(f"Model loaded! Vocabulary size: {len(model)} words")
print(f"Each word is a vector of {model.vector_size} numbers")

## 1. Words as Vectors

Each word in the model is represented as a vector of 300 numbers. Let's look at what this means.

In [2]:
# Look at the vector for one word
word = 'king'
vector = model[word]
print(f"Vector for '{word}': {vector[:10]}...")  # Show first 10 of 100 numbers
print(f"Shape: {vector.shape}")
print(f"Length (norm): {np.linalg.norm(vector):.2f}")

Vector for 'king': [-0.32307 -0.87616  0.21977  0.25268  0.22976  0.7388  -0.37954 -0.35307
 -0.84369 -1.1113 ]...
Shape: (100,)
Length (norm): 6.12


Each number represents something about the word's meaning --- but unlike our toy example with "royal", "gender", and "age" dimensions, the 100 dimensions don't have simple human-readable labels. The computer figured out its own useful dimensions from reading text.

## 2. Finding Similar Words

If two words have similar meanings, their vectors should be close together. We can find the most similar words to any word by searching for the nearest vectors.

In [3]:
# Find words most similar to a given query
query = 'soccer'
print(f"Most similar to '{query}':")
for word, score in model.most_similar(query):
    print(f"  {word}: {score:.3f}")

Most similar to 'soccer':
  football: 0.873
  basketball: 0.800
  volleyball: 0.769
  hockey: 0.744
  league: 0.742
  sports: 0.731
  club: 0.730
  rugby: 0.724
  team: 0.715
  tennis: 0.704


**TODO:** Pick 3--4 words from *different* categories (e.g., a food, a profession, an emotion, an animal) and use `model.most_similar()` to find the 5 nearest neighbors for each. Print the results in the same format as the cell above. Do the neighbors make sense? Do any of them surprise you?

In [ ]:
### BEGIN SOLUTION
for query in ['guitar', 'france', 'angry', 'dog']:
    results = model.most_similar(query, topn=5)
    top_words = [f"{w} ({s:.2f})" for w, s in results]
    print(f"Similar to '{query}': {', '.join(top_words)}")
    print()
### END SOLUTION

**Answer:** The model groups words into semantic categories it learned entirely from reading text --- nobody told it what a sport or a country is! For the example words above:
- "guitar" finds other **instruments** (bass, drums, harmonica)
- "france" finds other **countries** and related proper nouns (belgium, britain, spain, paris)
- "angry" finds other **emotions** (furious, outraged, enraged)
- "dog" finds other **animals/pets** (cat, puppy, horse)

Your own words should show similar clustering. If any neighbors seem surprising, consider what text contexts those words might share.

## 3. Analogies: Vector Arithmetic with Meaning

The most famous property of word embeddings is that you can do **arithmetic** with meaning.

The classic example: **king - man + woman = queen**

The idea: the vector from "man" to "king" captures the concept of "royalty". Adding that same direction to "woman" gives us "queen".

In [5]:
# Solve analogy: base - man + woman = ?
base = 'king'
result = model.most_similar(positive=[base, 'woman'], negative=['man'], topn=3)
print(f"{base} - man + woman = ?")
for word, score in result:
    print(f"  {word}: {score:.3f}")

king - man + woman = ?
  queen: 0.770
  monarch: 0.684
  throne: 0.676


**TODO:** Come up with 3--4 analogies of your own to test the model (e.g., capitals, slow:slower, verb tenses). Define them using the format we used above (giving positive and negative words) and use `model.most_similar()` to solve them.

Remember the format is: `result = model.most_similar(positive=[...], negative=[...], topn=...)`

In [ ]:
### BEGIN SOLUTION
# More analogies
analogies = [
    (['paris', 'germany'], ['france'], "paris : france :: ? : germany"),
    (['bigger', 'slow'], ['big'], "big : bigger :: slow : ?"),
    (['queen', 'boy'], ['king'], "king : queen :: boy : ?"),
]

for positive, negative, description in analogies:
    result = model.most_similar(positive=positive, negative=negative, topn=3)
    top = result[0]
    print(f"{description}")
    print(f"  Answer: {top[0]} ({top[1]:.3f})")
    print()
### END SOLUTION

**Answer:** The model learned:
- **Geography**: capital-country relationships
- **Grammar**: comparative forms (big/bigger, slow/slower)
- **Gender**: king/queen, boy/girl

All from reading text!

## 4. Combining Concepts

We can also **add** word vectors to combine concepts and see what the model finds.

In [7]:
# Combine two concepts. For each pair we show the top-5 matches
# and also the similarity score for a few words we might *expect* to see.
combinations = [
    (['spain', 'sports'],        ['soccer', 'tennis', 'basketball']),
    (['spanish', 'food'],        ['paella', 'tapas', 'wine']),
    (['japanese', 'cuisine'],    ['sushi', 'rice', 'noodle']),
    (['computer', 'music'],      ['synthesizer', 'recording', 'digital']),
]

for words, expected in combinations:
    # Normalized sum of the two word vectors
    combined = model[words[0]] + model[words[1]]
    combined = combined / np.linalg.norm(combined)

    # Top matches, excluding the input words themselves
    results = model.similar_by_vector(combined, topn=12)
    filtered = [(w, s) for w, s in results if w.lower() not in [x.lower() for x in words]][:8]
    top_str = ', '.join(f"{w} ({s:.2f})" for w, s in filtered)

    # Similarity for specific expected words
    exp_scores = []
    for e in expected:
        e_vec = model[e] / np.linalg.norm(model[e])
        sim = float(np.dot(combined, e_vec))
        exp_scores.append(f"{e} ({sim:.2f})")
    exp_str = ', '.join(exp_scores)

    print(f"{' + '.join(words)}:")
    print(f"  Top matches: {top_str}")
    print(f"  Expected:    {exp_str}")
    print()

spain + sports:
  Top matches: sport (0.74), soccer (0.73), sporting (0.70), world (0.68), italy (0.68), france (0.68), spanish (0.67), football (0.66)
  Expected:    soccer (0.73), tennis (0.59), basketball (0.57)

spanish + food:
  Top matches: french (0.73), italian (0.70), mexican (0.70), well (0.67), local (0.65), other (0.64), the (0.64), american (0.64)
  Expected:    paella (0.14), tapas (0.21), wine (0.56)

japanese + cuisine:
  Top matches: chinese (0.68), japan (0.63), traditional (0.63), korean (0.61), dishes (0.60), culture (0.60), taiwanese (0.60), popular (0.60)
  Expected:    sushi (0.58), rice (0.38), noodle (0.40)

computer + music:
  Top matches: software (0.77), electronic (0.75), video (0.75), digital (0.75), internet (0.73), computers (0.71), technology (0.71), online (0.70)
  Expected:    synthesizer (0.47), recording (0.70), digital (0.75)



**Question:** Compare the results for `japanese + cuisine` and `spanish + food`. The Japanese pairing finds actual dishes (sushi, rice, noodle) with decent scores, while the Spanish pairing returns mostly generic commodity words (foods, supplies, products). What two factors explain this difference?

**Answer:** Two factors are at play:

1. **"cuisine" and "food" mean different things in this corpus.** The top matches reveal it: `japanese + cuisine` surfaces *dishes, traditional, culture, korean* --- the cultural/culinary cluster. `spanish + food` surfaces *foods, supplies, products, grocery* --- the commodity/trade cluster. In English news and Wikipedia, "food" mostly describes imports and supplies; "cuisine" describes culinary traditions. The choice of anchor word controls which region of the embedding space you land in.

2. **English-language coverage favors Japanese food.** *Sushi* appears far more often in English text than *paella* does --- Japanese cuisine has a global restaurant presence and extensive English-language coverage. Rare words like *paella* get weaker, noisier vectors because the model has fewer co-occurrence examples to learn from. Embeddings reflect what the English corpus *talks about*, not what's culturally central to each country.

## 5. Bias in Embeddings

Word embeddings learn from real text, which contains real **biases**. Let's see an example.

In [ ]:
# For each profession: profession - man + woman = ?
for profession in ['doctor', 'engineer']:
    result = model.most_similar(positive=[profession, 'woman'], negative=['man'], topn=5)
    print(f"{profession} - man + woman = ?")
    for word, score in result:
        print(f"  {word}: {score:.3f}")
    print()

doctor - man + woman = ?
  mexican: 0.533
  hospital: 0.512
  nurse: 0.511
  doctors: 0.510
  ernesto: 0.509

engineer - man + woman = ?
  electrician: 0.509
  mexican: 0.508
  technician: 0.502
  herrera: 0.496
  architect: 0.494



The model often maps male professions to female professions rather than keeping the same profession. This reflects **gender stereotypes** in the training data (Google News articles).

This is an important lesson: **ML models learn patterns from data, including harmful biases.** We'll discuss this more in L21.

**TODO:** Further Exploration! Can you find 2--3 other words (e.g., other professions, adjectives like 'smart' or 'beautiful') that exhibit gender bias in the same way? Use the code pattern from above to test them.

In [ ]:
### BEGIN SOLUTION
for profession in ['programmer', 'receptionist', 'genius']:
    result = model.most_similar(positive=[profession, 'woman'], negative=['man'], topn=5)
    print(f"{profession} - man + woman = ?")
    for word, score in result:
        print(f"  {word}: {score:.3f}")
    print()
### END SOLUTION

## 6. The Limitation: One Word, One Vector

Word embeddings give each word a single, fixed vector. But many words have multiple meanings depending on context.

Think about the word "pound":
- "She paid 20 **pounds** for the ticket." (British currency)
- "The baby weighs 8 **pounds**." (unit of weight)
- "He will **pound** the nail into the wall." (verb: to hit)

In our model, "pound" has just one vector --- a blend of all its meanings.

In [9]:
# A polysemous word — "pound" has multiple meanings blended into one vector
query = 'pound'
print(f"Most similar to '{query}':")
for word, score in model.most_similar(query, topn=12):
    print(f"  {word}: {score:.3f}")

Most similar to 'pound':
  pounds: 0.733
  dollar: 0.629
  kilogram: 0.626
  cent: 0.625
  ounce: 0.624
  cents: 0.618
  euro: 0.611
  ounces: 0.599
  sterling: 0.584
  barrel: 0.578
  pence: 0.565
  franc: 0.564


You'll see a mix of **currency** words (*dollar*, *euro*, *sterling*, *pence*, *franc*) and **weight/unit** words (*kilogram*, *ounce*, *barrel*). The single vector is a compromise that doesn't fully capture any one meaning.

**The solution?** We need representations that **change based on context** --- different vectors for "pound" depending on the surrounding words. That's exactly what **attention mechanisms** and **transformers** do, which we'll cover next time.

**TODO:** Further Exploration! Find 2--3 other polysemous words (words with multiple meanings, e.g., 'bank', 'bark', 'bat', 'apple') and print their top 10-12 similar words. Do you see the different meanings blended together?

In [ ]:
### BEGIN SOLUTION
for query in ['bank', 'bark', 'bat']:
    print(f"Most similar to '{query}':")
    for word, score in model.most_similar(query, topn=10):
        print(f"  {word}: {score:.3f}")
    print()
### END SOLUTION

## Summary

1. **Word embeddings** map words to vectors (100 numbers in this GloVe model)
2. Learned from billions of words: similar contexts → similar vectors
3. **Similarity**: nearest vectors to "soccer" = other sports
4. **Analogies**: king - man + woman ≈ queen (vector arithmetic captures relationships)
5. **Composition**: adding vectors combines concepts (spain + sports → soccer, sporting)
6. **Bias warning**: embeddings reflect biases in training data
7. **Limitation**: one fixed vector per word, but meaning depends on context
8. **Next (L21)**: attention & transformers --- context-dependent representations